# 01 — 4クラスSER特徴cacheの準備

MSP-Podcast R1.10、HCUDB1、IEMOCAPのmetadata監査、version付きmapping/split、manifest、固定encoder出力のshard cacheを確認します。正式な全件処理は明示フラグを変更するまで開始しません。


In [ ]:
import os, sys, subprocess
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'ser_pipeline').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ser_pipeline.manifest import audit_dataset
from ser_pipeline.notebook_api import (
    demo_cache_summary, environment_summary, extraction_command_preview,
    mapping_summary, one_item_feature_benchmark, split_summary,
)

RUN_FULL_EXTRACTION = False
ARTIFACT_DIR = PROJECT_ROOT / 'runs' / 'ser_feature_preflight'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


## 1. 実行環境と固定契約


In [ ]:
environment_summary()


In [ ]:
pd.DataFrame(mapping_summary())


In [ ]:
split_summary()


## 2. ローカルmetadata監査

環境変数でrootが設定されたデータセットだけを読み取り専用で監査します。未設定時は空の表になり、demoは継続します。


In [ ]:
configured_roots = {
    'msp_podcast': os.environ.get('MSP_PODCAST_ROOT'),
    'hcudb1': os.environ.get('HCUDB1_ROOT'),
    'iemocap': os.environ.get('IEMOCAP_ROOT'),
}
audit_rows = [audit_dataset(name, root) for name, root in configured_roots.items() if root]
pd.DataFrame(audit_rows)


## 3. 合成manifest/cacheの境界確認


In [ ]:
demo_status = demo_cache_summary(ARTIFACT_DIR / 'demo')
demo_status


## 4. 1件preflightと容量表示


In [ ]:
one_item_feature_benchmark(feature_dim=768, seconds=1.0)


## 5. 全件コマンドのpreview

次のセルはコマンド文字列を表示するだけです。正式実行時もlayerは `final` 固定です。


In [ ]:
full_command = extraction_command_preview()
full_command


In [ ]:
if RUN_FULL_EXTRACTION:
    raise RuntimeError('Placeholders must be replaced and the formal preflight gate approved before execution')
else:
    {'status': 'preview_only', 'command': full_command}
